In [1]:
import numpy as np
import matplotlib.pyplot as plt
import io
import time
import random
import math
from robot_client import RobotClient, OscListener
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple
import cv2



from __future__ import annotations
import random
import argparse
import os
import signal
import sys
from pathlib import Path


import pygame

from src.config import load_config, set_seed
from src.environment import BraitenbergEnv
from src.pygame_view import PygameView

from src.config import load_config, set_seed

pygame 2.6.1 (SDL 2.32.56, Python 3.11.15)
Hello from the pygame community. https://www.pygame.org/contribute.html


<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
c:\Users\LoveTriangle\miniconda3\envs\rl\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
# cfg = load_config("config/config.yaml")
width = 180 #cfg['arena']['width']
height = 240 #cfg['arena']['height']

In [3]:
env = BraitenbergEnv(cfg, cfg.get("condition"))

NameError: name 'cfg' is not defined

In [2]:
# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)


In [3]:
device

device(type='cuda')

In [9]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# Model

In [4]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [5]:
class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [9]:
  
# def grab_and_build_state(BraitenbergEnv):
#     x = BraitenbergEnv.vehicle.x # need to normalize this
#     y = BraitenbergEnv.vehicle.y
#     x_norm = x/width 
#     y_norm = y/height
#     dt = BraitenbergEnv.step.dt
#     last_x= BraitenbergEnv.last_info.vehicle_x
#     last_y = BraitenbergEnv.last_info.vehicle_y
#     agent_vel = np.sqrt((x-last_x)**2 + (y-last_y)**2)/dt
#     heading_theta = BraitenbergEnv.vehicle.heading
#     sin_theta = np.sin(heading_theta)
#     cos_theta = np.cos(heading_theta)
#     agent_vel = BraitenbergEnv.last_info.
#     red_x, red_y = BraitenbergEnv.red_pos
#     green_x, green_y = BraitenbergEnv.green_pos


#     return np.array([x_norm, y_norm, sin_theta, cos_theta, agent_vel, green_x, green_y,  red_x, red_y])

# Hyperparamters 

In [10]:
# BATCH_SIZE is the number of transitions sampled from the replay buffer
# GAMMA is the discount factor as mentioned in the previous section
# EPS_START is the starting value of epsilon
# EPS_END is the final value of epsilon
# EPS_DECAY controls the rate of exponential decay of epsilon, higher means a slower decay
# TAU is the update rate of the target network
# LR is the learning rate of the ``AdamW`` optimizer

BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 50000
TAU = 0.005
LR = 3e-4


# Get number of actions from gym action space
n_actions = len(env.make_action())
# Get the number of state observations
state,info = env.reset()

n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)


steps_done = 0






# Load Checkpoint (optional)

In [11]:
def select_action(state):
    global steps_done

    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(
        -steps_done / EPS_DECAY
    )
    steps_done += 1

    if sample > eps_threshold:
        with torch.no_grad():
            action_idx = policy_net(state).max(1).indices.view(1, 1)
            # print("action index:", action_idx.item())
            return action_idx
    else:
        action_idx = random.randrange(len(env.action_space))
        return torch.tensor([[action_idx]], device=device, dtype=torch.long)
    
episode_durations = []
episode_rewards = []

In [12]:
def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    # is_ipython = True
    # if is_ipython:
    #     if not show_result:
    #         display.display(plt.gcf())
    #         display.clear_output(wait=True)
    #     else:
    #         display.display(plt.gcf())

In [13]:
def plot_rewards(show_result=False):
    plt.figure(2)
    rewards_t = torch.tensor(episode_rewards, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Total reward')
    plt.plot(rewards_t.numpy())
    # Take 100 episode averages and plot them too
    if len(rewards_t) >= 100:
        means = rewards_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated

In [14]:
def run_rollout(policy_net, env, max_steps=None, device=device):
    """Greedy rollout of a trained policy_net in BraitenbergEnv (no exploration)."""
    policy_net.eval()
    max_steps = max_steps or cfg['simulation']['max_steps']

    state, info = env.reset()
    state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    trajectory = {'x': [], 'y': [], 'red_x': [], 'red_y': [], 'green_x': [], 'green_y': []}
    rewards = []

    for t in range(max_steps):
        with torch.no_grad():
            action = policy_net(state_t).max(1).indices.view(1, 1)

        observation, reward, terminated = env.step(action)
        rewards.append(reward)
        trajectory['x'].append(env.vehicle.state.x)
        trajectory['y'].append(env.vehicle.state.y)
        trajectory['red_x'].append(env.red_pos[0])
        trajectory['red_y'].append(env.red_pos[1])
        trajectory['green_x'].append(env.green_pos[0])
        trajectory['green_y'].append(env.green_pos[1])

        if terminated:
            break

        state_t = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

    policy_net.train()
    return trajectory, rewards


def plot_rollout(trajectory, rewards, title_prefix=""):
    fig, ax = plt.subplots()
    ax.plot(trajectory['x'], trajectory['y'], 'o-', color='blue', markersize=3, label='rollout')
    ax.plot(trajectory['x'][0], trajectory['y'][0], 's', color='black', markersize=10, label='start')
    ax.plot(trajectory['x'][-1], trajectory['y'][-1], '*', color='red', markersize=15, label='end')
    ax.plot(trajectory['green_x'][-1], trajectory['green_y'][-1], '^', color='green', markersize=10, label='green target')
    ax.plot(trajectory['red_x'][-1], trajectory['red_y'][-1], 'v', color='darkred', markersize=10, label='red (avoid)')
    half_w = cfg['arena']['width'] / 2
    half_h = cfg['arena']['height'] / 2
    ax.set_xlim(-half_w, half_w)
    ax.set_ylim(half_h, -half_h)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f"{title_prefix}Rollout — total reward: {sum(rewards):.2f}, steps: {len(rewards)}")
    ax.legend()
    plt.show()

# Training

In [1]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
    # detailed explanation). This converts batch-array of Transitions
    # to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    # (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1).values
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [ ]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 10000
else:
    num_episodes = 50

ROLLOUT_EVERY = 10

for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    episode_reward = 0.0
    for t in range(cfg['simulation']['max_steps']):
        action = select_action(state)
        observation, reward, terminated = env.step(action)
        episode_reward += reward
        reward = torch.tensor([reward], device=device)
        done = terminated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            episode_rewards.append(episode_reward)
            break

    if (i_episode + 1) % ROLLOUT_EVERY == 0:
        plot_rewards()
        rollout_trajectory, rollout_rewards = run_rollout(policy_net, env)
        plot_rollout(rollout_trajectory, rollout_rewards, title_prefix=f"Episode {i_episode + 1} — ")

print('Complete')
plot_rewards(show_result=True)
plt.ioff()
plt.show()

In [17]:
# redo old functions to work with continuous state space




# def get_position(state):
#     """Convert a flat state index into (x, y, orientation).
#     Origin (0, 0) is the top-left corner; y increases downward.
#     State indices run row-major over (x, y), with orientation as the
#     fastest-varying component: state = (y * N_COLS + x) * N_ORIENTATIONS + orientation_idx
#     """
#     total_states = N_COLS * N_ROWS * N_ORIENTATIONS
#     if not (0 <= state < total_states):
#         raise ValueError(f"state {state} is out of bounds for a {N_COLS}x{N_ROWS}x{N_ORIENTATIONS} state space")
#     o_idx = state % N_ORIENTATIONS
#     grid_idx = state // N_ORIENTATIONS
#     x = grid_idx % N_COLS
#     y = grid_idx // N_COLS
#     return x, y, ORIENTATIONS[o_idx]

# Utilities for Real-World

In [5]:
def list_checkpoints(root="outputs/dqn_runs", n=10):
    """Return up to n most-recently-modified checkpoint files under root
    (newest first), looking in both checkpoints/*.pt and policies/*.pt
    subfolders of every run directory.
    """
    root = Path(root)
    paths = list(root.glob("**/checkpoints/*.pt")) + list(root.glob("**/policies/*.pt"))
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[:n]


def print_checkpoints(root="outputs/dqn_runs", n=10):
    """Print an indexed list of the n most recently-modified checkpoints, so
    you can pick an index to pass to load_checkpoint()."""
    paths = list_checkpoints(root, n)
    for i, p in enumerate(paths):
        print(f"[{i}] {p}")
    return paths


def load_checkpoint(selection, policy_net, target_net=None, root="outputs/dqn_runs", device=device):
    """Load a checkpoint into policy_net (and target_net, if given) in place.
    selection: index into list_checkpoints() (0 = most recently modified), or
    a direct path to a .pt file (from print_checkpoints() or elsewhere).
    Returns the raw checkpoint payload (episode, rewards, config, etc.).

    Uses weights_only=False: these checkpoints carry a plain-dict "config"
    alongside the tensors, which torch's weights_only safe-unpickler (default
    since torch 2.6) can't load — and on this torch 2.12 install, the
    weights_only path even fails outright with
    AttributeError: module 'torch' has no attribute '_utils'
    before it gets to the unsupported-type check. weights_only=False sidesteps
    both issues via plain pickle; safe here since these are our own training
    checkpoints, not files from an untrusted source.
    """
    if isinstance(selection, (str, Path)):
        path = Path(selection)
    else:
        candidates = list_checkpoints(root, n=selection + 1)
        if selection >= len(candidates):
            raise FileNotFoundError(f"Only found {len(candidates)} checkpoint(s) under {root}")
        path = candidates[selection]

    payload = torch.load(path, map_location=device, weights_only=False)
    state_dict = payload.get("policy_state_dict")
    if state_dict is None:
        raise KeyError(f"No policy_state_dict found in {path}")

    policy_net.load_state_dict(state_dict)
    if target_net is not None and "target_state_dict" in payload:
        target_net.load_state_dict(payload["target_state_dict"])

    episode_note = f" (episode {payload['episode']})" if "episode" in payload else ""
    print(f"Loaded checkpoint: {path}{episode_note}")
    return payload


def make_action_space(cfg):
    """Same enumeration as BraitenbergEnv.make_action, computed directly from
    cfg so we don't need a live env instance just to know how many actions
    there are.
    """
    max_speed = int(cfg["vehicle"]["max_linear_speed"])
    speeds = range(-max_speed, max_speed + 1, 20)
    return [[i, j] for i in speeds for j in speeds]


def build_policy_nets(cfg, device=device, hidden_dim=128, n_observations=9):
    """Construct fresh policy_net/target_net sized for this cfg, without
    instantiating BraitenbergEnv or calling reset(). n_observations defaults
    to 9 — BraitenbergEnv.build_state always returns a 9-dim vector regardless
    of env state (see BraitenbergEnv.state_dim). n_actions is derived from
    cfg['vehicle']['max_linear_speed'] via make_action_space, matching
    BraitenbergEnv.make_action()'s enumeration without needing the env.
    """
    n_actions = len(make_action_space(cfg))
    policy_net = DQN(n_observations, n_actions).to(device)
    target_net = DQN(n_observations, n_actions).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    return policy_net, target_net


# example usage:
# print_checkpoints()                              # see what's available, pick an index
# policy_net, target_net = build_policy_nets(cfg)  # no env.reset() needed
# load_checkpoint(0, policy_net, target_net)       # 0 = most recently modified

NameError: name 'device' is not defined

In [ ]:
# arena corners in raw camera pixel coordinates (calibrated once, camera isn't axis-aligned with the arena)
# fallback default — run_rollout_REAL builds a fresh homography per-run from
# Bonsai's live /corners12 + /corners34 OSC feed instead of using this constant
ARENA_CORNERS_PX = np.array([
    [1217, 400],  # top left
    [157, 890],   # top right
    [1159, 71],   # bottom left
    [179, 172],   # bottom right
], dtype=np.float32)

# corresponding corners in our state-space coordinate frame, center is 0,0
ARENA_CORNERS_STATE = np.array([
    [height-1 /2 , -width-1/2],
    [height-1 /2 , width-1/2],
    [-height-1 /2, -width-1/2],
    [-height-1/2, width - 1/2],
], dtype=np.float32)


def build_arena_homography(corners_px):
    """Build the pixel<->arena homography from 4 corners in raw camera pixel
    coordinates, ordered [top-left, top-right, bottom-left, bottom-right] —
    same order as ARENA_CORNERS_PX/ARENA_CORNERS_STATE. Returns (homography,
    inverse_homography) for use with pixel_to_arena/arena_to_pixel.
    """
    corners_px = np.asarray(corners_px, dtype=np.float32)
    homography = cv2.getPerspectiveTransform(corners_px, ARENA_CORNERS_STATE)
    homography_inv = cv2.getPerspectiveTransform(ARENA_CORNERS_STATE, corners_px)
    return homography, homography_inv


_ARENA_HOMOGRAPHY, _ARENA_HOMOGRAPHY_INV = build_arena_homography(ARENA_CORNERS_PX)


def pixel_to_arena(px, py, homography=None):
    """Map a raw camera pixel coordinate to (x, y) in the state-space coordinate frame."""
    pt = np.array([[[px, py]]], dtype=np.float32)
    mapped = cv2.perspectiveTransform(pt, homography if homography is not None else _ARENA_HOMOGRAPHY)
    return float(mapped[0, 0, 0]), float(mapped[0, 0, 1])


def arena_to_pixel(x, y, homography_inv=None):
    """Map a state-space (x, y) coordinate back to raw camera pixel coordinates
    (the inverse of pixel_to_arena), e.g. for overlaying the goal region on the
    live camera frame.
    """
    pt = np.array([[[x, y]]], dtype=np.float32)
    mapped = cv2.perspectiveTransform(pt, homography_inv if homography_inv is not None else _ARENA_HOMOGRAPHY_INV)
    return float(mapped[0, 0, 0]), float(mapped[0, 0, 1])


def vector_to_orientation(dx, dy):
    """Heading angle (radians) of the green->red vector, in the same convention
    as the simulation: heading=0 points along +x, increasing counter-clockwise,
    so x += cos(heading), y += sin(heading) (see DifferentialDriveVehicle.update).
    dx, dy must already be in arena state-space coordinates (i.e. after
    pixel_to_arena), not raw camera pixels.
    """
    return math.atan2(dy, dx)


def has_invalid_marker(robot_data):
    """Bonsai sends -1 for any marker (red/green) it couldn't detect in the current frame."""
    return any(v == -1 for v in robot_data)


def robot_to_state(robot_data, red_data, green_data, prev_xy=None, dt=None, homography=None):
    """Convert the robot's two-marker camera readout into the same 9-dim state
    vector used by the DQN in simulation (see BraitenbergEnv.build_state):
    [x_norm, y_norm, sin_theta, cos_theta, norm_vel, norm_green_x, norm_green_y, norm_red_x, norm_red_y].
    robot_data: [x_red, y_red, x_green, y_green] markers mounted on our robot
    (front/back), in raw camera pixel coordinates — used for position + heading.
    red_data: [x, y] raw pixel position of the red (avoid) stimulus robot.
    green_data: [x, y] raw pixel position of the green (target) stimulus robot.
    prev_xy/dt: previous arena-frame (x, y) and elapsed time, for velocity; if
    omitted (e.g. first step) norm_vel is reported as 0.
    homography: optional pixel->arena homography (see build_arena_homography);
    defaults to the hardcoded ARENA_CORNERS_PX calibration if omitted.
    """
    x_red_px, y_red_px, x_green_px, y_green_px = robot_data
    x_red_aux_px, y_red_aux_px = red_data
    x_green_aux_px, y_green_aux_px = green_data

    # stimulus robots (red = avoid, green = target)
    x_aux_red, y_aux_red = pixel_to_arena(x_red_aux_px, y_red_aux_px, homography)
    x_aux_green, y_aux_green = pixel_to_arena(x_green_aux_px, y_green_aux_px, homography)

    # our robot's own front/back markers -> position + heading
    x_red, y_red = pixel_to_arena(x_red_px, y_red_px, homography)
    x_green, y_green = pixel_to_arena(x_green_px, y_green_px, homography)

    heading = vector_to_orientation(x_red - x_green, y_red - y_green)  # radians, matches sim's heading convention
    sin_theta = math.sin(heading)
    cos_theta = math.cos(heading)

    half_w = width / 2.0
    half_h = height / 2.0

    x_norm = x_red / half_w
    y_norm = y_red / half_h

    if prev_xy is None or dt is None:
        norm_vel = 0.0
    else:
        prev_x, prev_y = prev_xy
        agent_vel = np.sqrt((x_red - prev_x) ** 2 + (y_red - prev_y) ** 2) / dt
        norm_vel = agent_vel / float(cfg['vehicle']['max_linear_speed'])

    norm_red_x = x_aux_red / half_w
    norm_red_y = y_aux_red / half_h
    norm_green_x = x_aux_green / half_w
    norm_green_y = y_aux_green / half_h

    return np.array(
            [
                x_norm,
                y_norm,
                sin_theta,
                cos_theta,
                norm_vel,
                norm_green_x,
                norm_green_y,
                norm_red_x,
                norm_red_y,
            ],
            dtype=np.float32,
        )


def reward_from_positions(vehicle_xy, red_xy, green_xy):
    """Mirrors BraitenbergEnv.reward_function (src/environment.py), but computed
    directly from raw (unnormalized) arena-frame positions instead of env state,
    since there's no BraitenbergEnv instance stepping on real hardware.
    """
    vehicle_pos = np.array(vehicle_xy)
    red_pos = np.array(red_xy)
    green_pos = np.array(green_xy)

    half_w = width / 2.0
    half_h = height / 2.0

    distance_to_green = np.linalg.norm(vehicle_pos - green_pos)
    distance_to_red = np.linalg.norm(vehicle_pos - red_pos)
    distance_to_boundaries = min(
        vehicle_pos[0] + half_w,   # distance from left wall
        half_w - vehicle_pos[0],   # distance from right wall
        vehicle_pos[1] + half_h,   # distance from bottom wall
        half_h - vehicle_pos[1],   # distance from top wall
    )

    reward = 0.0
    if distance_to_green < 10:  # Threshold for being very close to the green robot
        reward += 100.0
    elif distance_to_boundaries < 5:  # Threshold for being very close to the boundaries
        reward -= 5.0 * 1 / distance_to_boundaries
    if distance_to_red < 10:  # Threshold for being very close to the red robot
        reward -= 100.0
    reward -= 0.1 * distance_to_green
    reward += 0.05 * distance_to_red

    return reward


In [ ]:
def run_rollout_REAL(policy_net, env, max_steps=None, device=device, position_timeout=5.0):
    """Greedy rollout of a trained policy_net in BraitenbergEnv (no exploration)."""
    robot = RobotClient()
    robot.start()
    policy_net.eval()
    robot_data = [0, 0, 0, 0]  # [x_red, y_red, x_green, y_green] (raw camera pixels, our robot's markers)
    aux_data = [0, 0, 0, 0]  # [x_red, y_red, x_green, y_green] (raw camera pixels, stimulus robots)
    corners = [None, None, None, None]  # [top-left, top-right, bottom-left, bottom-right] (raw camera pixels)
    got_position = False
    got_aux = False

    def on_robot(args):
        nonlocal got_position
        robot_data[:] = args
        got_position = True

    def on_auxrobots(args):
        nonlocal got_aux
        aux_data[:] = args
        got_aux = True

    def on_corners12(args):
        corners[0] = (args[0], args[1])
        corners[1] = (args[2], args[3])

    def on_corners34(args):
        corners[2] = (args[0], args[1])
        corners[3] = (args[2], args[3])

    positions = OscListener(port=9000)
    positions.subscribe("/robot", on_robot)
    positions.subscribe("/auxrobots", on_auxrobots)
    positions.subscribe("/corners12", on_corners12)
    positions.subscribe("/corners34", on_corners34)
    positions.start()
    
    waited = 0.0
    while not (got_position and got_aux and all(c is not None for c in corners)) and waited < position_timeout:
        time.sleep(0.1)
        waited += 0.1
    if not (got_position and got_aux and all(c is not None for c in corners)):
        robot.stop()
        raise RuntimeError(f"Timed out after {position_timeout}s waiting for /robot, /auxrobots and /corners12+/corners34 from Bonsai")

    if has_invalid_marker(robot_data) or has_invalid_marker(aux_data):
        robot.stop()
        raise RuntimeError("Marker not detected at start — check camera/tracking before running.")

    homography, _ = build_arena_homography(corners)  # live calibration, replaces the hardcoded ARENA_CORNERS_PX

    half_w = width / 2.0
    half_h = height / 2.0
    dt = cfg['simulation']['dt']
    action_space = make_action_space(cfg)  # [left, right] motor pairs, same enumeration as BraitenbergEnv.make_action
    red_data, green_data = aux_data[:2], aux_data[2:]
    state = robot_to_state(robot_data, red_data, green_data, homography=homography)  # uses real pixel coords to grab state for policy
    prev_xy = (state[0] * half_w, state[1] * half_h)

    max_steps = max_steps or cfg['simulation']['max_steps']

    state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    trajectory = {'x': [], 'y': [], 'red_x': [], 'red_y': [], 'green_x': [], 'green_y': []}
    rewards = []
    terminated = False

    for t in range(max_steps):

        with torch.no_grad():
            action_idx = policy_net(state_t).max(1).indices.item()

        # the DQN outputs an action index, not motor values directly — look it
        # up in action_space the same way BraitenbergEnv.step does for
        # "Case 1: DQN action index" (see environment.py)
        left, right = action_space[action_idx]
        duration = dt

        robot.set_wheels(left, right)  # drive
        print(f"State: {state}, Action idx: {action_idx}, Wheels: (L={left}, R={right}), Duration: {duration}s")
        time.sleep(duration)  # pauses

        robot.set_wheels(0, 0)  # stop moving before reading position
        red_data, green_data = aux_data[:2], aux_data[2:]
        if has_invalid_marker(robot_data) or has_invalid_marker(aux_data):
            print("Warning: marker not detected this step, keeping previous state.")
        else:
            state = robot_to_state(robot_data, red_data, green_data, prev_xy=prev_xy, dt=dt, homography=homography)
            prev_xy = (state[0] * half_w, state[1] * half_h)

        vehicle_xy = (state[0] * half_w, state[1] * half_h)
        green_xy = (state[5] * half_w, state[6] * half_h)
        red_xy = (state[7] * half_w, state[8] * half_h)

        trajectory['x'].append(vehicle_xy[0])
        trajectory['y'].append(vehicle_xy[1])
        trajectory['red_x'].append(red_xy[0])
        trajectory['red_y'].append(red_xy[1])
        trajectory['green_x'].append(green_xy[0])
        trajectory['green_y'].append(green_xy[1])

        reward = reward_from_positions(vehicle_xy, red_xy, green_xy)
        rewards.append(reward)

        # mirrors the sim's reward shaping — a hit pushes the reward to roughly
        # +/-100, so we treat that as a stand-in for "terminated" on real hardware
        if reward >= 100 or reward <= -100:
            terminated = True
            break

        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    print("Goal reached!" if terminated and rewards[-1] >= 100 else
          "Hit red — terminated." if terminated else
          "Max steps reached without reaching goal.")
    robot.set_wheels(0, 0)
    robot.stop()
    positions.stop()

    policy_net.train()
    return trajectory, rewards

In [ ]:
# aux = [red_x, red_y, green_x, green_y] 

In [ ]:
# Load a specific policy/checkpoint by path and run it on the real robot.
policy_path = "C:\\Users\\LoveTriangle\\Documents\\LoveTriangle_robots\\outputs\\dqn_runs\\config_cond-multi-blocking-orbit-separated_seed-7_training.lr=0.0001_training.eps_decay=20000_training.gamma=0.99_vehicle.max_linear_speed=20.0_simulation.dt=0.08\\multi-blocking-orbit-separated\\20260624_125026\\policies\\final_policy.pt"  # set this to the .pt you want to run
cfg = load_config("config/config.yaml")  # or the config used to train the policy, if different
env = BraitenbergEnv(cfg, cfg.get("condition"))  # make a fresh env for the rollout
policy_net, target_net = build_policy_nets(cfg)  # sized from cfg directly, no env.reset() needed
load_checkpoint(policy_path, policy_net, target_net)
trajectory, rewards = run_rollout_REAL(policy_net, env)
plot_rollout(trajectory, rewards, title_prefix="Real robot — ")

NameError: name 'build_policy_nets' is not defined